# 01 — Data validation

This notebook reviews the processed PhonePe Pulse tables and the validation artifacts produced by `phonepe_analytics.validate`. It does not repeat ETL logic.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
PROCESSED = ROOT / "data" / "processed"
state = pd.read_parquet(PROCESSED / "state_quarter.parquet")
district = pd.read_parquet(PROCESSED / "district_quarter.parquet")
checks = pd.read_csv(PROCESSED / "quality_checks.csv")
reconciliation = pd.read_csv(PROCESSED / "geographic_reconciliation.csv")
print(f"State-quarter rows: {len(state):,}")
print(f"District-quarter rows: {len(district):,}")
display(checks)

State-quarter rows: 1,224
District-quarter rows: 26,622


,check,table,failures
0,duplicate_logical_keys,state,0
1,null_state,state,0
2,invalid_year,state,0
3,invalid_quarter,state,0
4,negative_transaction_count,state,0
5,negative_transaction_amount,state,0
6,negative_registered_users,state,0
7,negative_registered_merchants,state,0
8,latest_core_values_complete,state,0
9,duplicate_logical_keys,district,0


The analytical grain is one geography in one calendar quarter. The six historical district rows without transaction records and the historical merchant gaps are kept as missing values. They are not converted to zero.

In [2]:
missing = pd.DataFrame(
    {
        "state_missing": state.isna().sum(),
        "district_missing": district.isna().sum(),
    }
)
display(missing[missing.sum(axis=1) > 0])
assert checks["failures"].eq(0).all()
assert reconciliation["transaction_count_difference"].eq(0).all()
assert reconciliation["registered_users_difference"].eq(0).all()

,state_missing,district_missing
registered_merchants,46.0,1885
transaction_amount,0.0,6
transaction_count,0.0,6


All 23 formal checks pass. Transaction counts, users, and available merchant counts reconcile between district and state cuts. Transaction amounts differ only by sub-rupee floating-point precision and remain below the one-rupee tolerance.